# Day 13 — Functional tools: itertools, functools, map/filter, lambda
Objectives:
- Use itertools for iterables.
- functools.lru_cache and partial.
- When to prefer vectorization over map/filter in DS code.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-13`. Read
`python/ds-60day/companion-guides/day13_functional_tools.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Python functions are objects: they can be assigned to names, passed as
arguments, and returned. A higher-order function accepts or returns
another function. This enables small reusable operations, but an
ordinary named function is usually clearer than a dense lambda once
logic needs explanation.

`map` transforms, `filter` selects, `itertools` composes lazy iteration,
and `functools` supplies function adapters and caching. List
comprehensions often read more naturally for one transformation/filter;
lazy tools matter when the input is a stream or too large to materialize.
Avoid `reduce` when a named loop or built-in such as `sum`, `min`, or
`max` states the intent better.

### Vocabulary

- **first-class function:** a function usable as an ordinary runtime value.
- **higher-order function:** a function that accepts or returns functions.
- **lambda:** a small anonymous single-expression function.
- **lazy iterator:** an iterator that computes values only when requested.
- **accumulator:** the progressively combined value in a fold/reduction.
- **cache:** stored results reused for repeated equivalent calls.

## Syntax anatomy

`map(normalize, names)` receives the function object `normalize`—without
parentheses—and an iterable. Iteration later calls it for each item.
`sorted(records, key=lambda row: row["score"])` calls the key function
once per row to derive comparison keys; the lambda returns one
expression.

### Worked example 1 — Pass a named function into a lazy transformation

Keep the operation testable on its own. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
def normalize_name(text: str) -> str:
    return " ".join(text.strip().title().split())

raw_names = ["  ada lovelace", "GRACE   HOPPER  "]
normalized_iter = map(normalize_name, raw_names)
list(normalized_iter)

**Expected observation:** `['Ada Lovelace', 'Grace Hopper']`. `map` is lazy; `list` consumes it.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Compose lazy filtering and slicing

Generate only as many values as the consumer requests. Predict first; then run the next cell.

In [ ]:
from itertools import islice

squares = (number**2 for number in range(100))
even_squares = filter(lambda value: value % 2 == 0, squares)
list(islice(even_squares, 5))

**Expected observation:** `[0, 4, 16, 36, 64]`. `islice` stops after five accepted values rather than consuming all 100 squares.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. If a function runs too early, check whether you passed `function` or called `function(...)`.
2. If output prints as a map/filter object, remember these are lazy iterators and consume only as needed.
3. Replace a multi-step lambda with a named function and tests.
4. Cache only functions whose result depends entirely on hashable arguments and stable external state.

**Alternative to compare:** Use a comprehension for a clear transform/filter, `itertools` for lazy composition, and an explicit loop when stateful branching matters.

**Boundary to test:** One-shot iterators, infinite inputs, side effects inside transformations, unhashable cache arguments, and empty reductions require care.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import itertools as it
from functools import lru_cache, partial

list(it.accumulate([1,2,3,4]))

@lru_cache(maxsize=None)
def fib(n:int)->int:
    return n if n<2 else fib(n-1)+fib(n-2)

fib(30)


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Use `itertools.groupby` to group records that are already sorted by a category key. **Inputs:** include the same category in separated positions before sorting.
   **Expected behavior:** after explicit sorting, each category appears once with all of its records. **Constraint:** explain why `groupby` groups adjacent runs rather than globally collecting unsorted data.
   **Verify:** Show the unsorted input produces separated runs, then assert the sorted grouping has one entry per category and preserves every record.

2. Use `functools.reduce` to compute a product for `[2, 3, 4]`, giving an explicit identity so empty input returns `1`. **Then:** implement the same result with a named loop and compare readability.
   **Verify:** both return `24` and both define empty behavior; state why `sum`/`math.prod` is preferable in ordinary production code.

### Additional mastery practice

Use functional tools when they make data flow clearer. Preserve laziness intentionally and keep side effects at explicit boundaries.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict what remains after calling `next` on a `map` object and then converting it to a list.
   **Progressive hint:** `map` is a lazy one-shot iterator in Python 3.
   **Verify:** Assert the first mapped value and exact remaining list, then confirm another pass is empty because the map iterator is exhausted.
4. **Tracing:** Trace `sorted(records, key=lambda row: (row['team'], -row['score']))` and explain the tuple key.
   **Progressive hint:** Tuple components are compared left to right.
   **Verify:** Compute each tuple key beside its record and assert final order groups team ascending and score descending within team.
5. **Implementation:** Implement `compose(*functions)` so `compose(f, g)(x)` applies `g` then `f`, and handle no functions as identity.
   **Progressive hint:** Apply the reversed function sequence to the current value.
   **Verify:** Assert composition order with noncommutative functions and assert `compose()(value)` returns the original value unchanged.
6. **Debugging:** Repair lambdas created in a loop that all use the final loop value.
   **Progressive hint:** Bind the current value as a default argument or use a factory function.
   **Verify:** Call every produced function and show the faulty results share the final loop value; assert the factory/default-binding repair preserves each intended value.
7. **Edge case and explanation:** Refactor a pipeline that prints inside `map` into pure transforms plus one explicit presentation step.
   **Progressive hint:** Pure stages are easier to test and reuse.
   **Verify:** Assert transformed values are identical before/after refactoring and capture presentation output only in the final explicit step.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Use `itertools.groupby` to group records that are already sorted by a category key. **Inputs:** include the same category in separated positions before sorting. **Expected behavior:** after explicit sorting, each category appears once with all of its records. **Constraint:** explain why `groupby` groups adjacent runs rather than globally collecting unsorted data. **Verify:** Show the unsorted input produces separated runs, then assert the sorted grouping has one entry per category and preserves every record.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Use `itertools.groupby` to group records that are already sorted by a category key. include the same category in separated positions before sorting. after explicit sorting, each...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Use `functools.reduce` to compute a product for `[2, 3, 4]`, giving an explicit identity so empty input returns `1`. **Then:** implement the same result with a named loop and compare readability. **Verify:** both return `24` and both define empty behavior; state why `sum`/`math.prod` is preferable in ordinary production code.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Use `functools.reduce` to compute a product for `[2, 3, 4]`, giving an explicit identity so empty input returns `1`. implement the same result with a named loop and compare read...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict what remains after calling `next` on a `map` object and then converting it to a list. **Progressive hint:** `map` is a lazy one-shot iterator in Python 3. **Verify:** Assert the first mapped value and exact remaining list, then confirm another pass is empty because the map iterator is exhausted.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict what remains after calling `next` on a `map` object and then converting it to a list. `map` is a lazy one-shot iterator in Python 3. Assert the first mapped value and ex...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace `sorted(records, key=lambda row: (row['team'], -row['score']))` and explain the tuple key. **Progressive hint:** Tuple components are compared left to right. **Verify:** Compute each tuple key beside its record and assert final order groups team ascending and score descending within team.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace `sorted(records, key=lambda row: (row['team'], -row['score']))` and explain the tuple key. Tuple components are compared left to right. Compute each tuple key beside its r...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement `compose(*functions)` so `compose(f, g)(x)` applies `g` then `f`, and handle no functions as identity. **Progressive hint:** Apply the reversed function sequence to the current value. **Verify:** Assert composition order with noncommutative functions and assert `compose()(value)` returns the original value unchanged.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Implement `compose(*functions)` so `compose(f, g)(x)` applies `g` then `f`, and handle no functions as identity. Apply the reversed function sequence to the current value. Asser...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair lambdas created in a loop that all use the final loop value. **Progressive hint:** Bind the current value as a default argument or use a factory function. **Verify:** Call every produced function and show the faulty results share the final loop value; assert the factory/default-binding repair preserves each intended value.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair lambdas created in a loop that all use the final loop value. Bind the current value as a default argument or use a factory function. Call every produced function and show...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Refactor a pipeline that prints inside `map` into pure transforms plus one explicit presentation step. **Progressive hint:** Pure stages are easier to test and reuse. **Verify:** Assert transformed values are identical before/after refactoring and capture presentation output only in the final explicit step.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Refactor a pipeline that prints inside `map` into pure transforms plus one explicit presentation step. Pure stages are easier to test and reuse. Assert transformed values are id...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
